# Managing Kleio files

> First time use: follow instructions in the [README.md](README.md) file in this directory.

## Initial setup

Create a TimelinkNotebook object. This will be the interface for most of your interactions with Timelink.

The first time you run this notebook, you will be prompted to install ìpykernel. This is required to run Timelink in a notebook.

Note:
* First time run takes a little time as the required Docker images are downloaded.
* Timelink will default to using sqlite as the database, see [Receipts notebook for more control](1-receipts.ipynb)


In [6]:
from timelink.notebooks import TimelinkNotebook
from timelink.kleio import KleioServer

# Alternative: attach to existing Kleio server for debugging
# kserver = KleioServer.attach(url='http://127.0.0.1:8088',token='mytoken',kleio_home='../sources')
# tlnb = TimelinkNotebook(kleio_server=kserver)
tlnb = TimelinkNotebook(kleio_image="kleio-server", 
                        kleio_version="12.9.592",
                        kleio_update=False,)

tlnb.print_info()


Timelink version: 1.2.0
Project name: dehergne-repertoire-mbook-air
Project home: /Users/jrc/develop/dehergne-repertoire-mbook-air
Database type: sqlite
Database name: dehergne_repertoire_mbook_air
Kleio image: kleio-server
Kleio server token: 47JF2...
Kleio server URL: http://127.0.0.1:8088
Kleio server home: /Users/jrc/develop/dehergne-repertoire-mbook-air
Kleio server container: optimistic_herschel
Kleio version requested: 12.9.592
Kleio server container status: running
Kleio server version: 12.9.592 (2026-03-19 23:14:02)
SQLite directory: /Users/jrc/develop/dehergne-repertoire-mbook-air/database/sqlite
Database version: 6ccf1ef385a6
Call print_info(show_token=True) to show the Kleio Server token
Call print_info(show_password=True) to show the Postgres password
TimelinkNotebook(project_name=dehergne-repertoire-mbook-air, project_home=/Users/jrc/develop/dehergne-repertoire-mbook-air, db_type=sqlite, db_name=dehergne_repertoire_mbook_air, kleio_image=kleio-server, kleio_version=12.9.5

TimelinkNotebook created a database to store data and a Kleio server to 
translate transcriptions in Kleio notation into data that can be imported
into a database.

To get information about the database and the Kleio server do ```tnlb.print_info()```

### Database status

Count the number of rows in each table in the database.


In [7]:
tlnb.table_row_count_df()

,table,count
0,acts,28
1,alembic_version,1
2,aregisters,0
3,attributes,26899
4,blinks,120
5,class_attributes,58
6,classes,10
7,entities,33057
8,geoentities,359
9,kleiofiles,35


### Show the kleio files available 

In [8]:
kleio_files = tlnb.get_kleio_files(match_path=True)
# kleio_files.info()
kleio_files[['name','status','errors','warnings','import_status','import_errors']].sort_values("name")  


,name,status,errors,warnings,import_status,import_errors
0,dehergne-0-abrev.cli,V,0,0,I,0
1,dehergne-a.cli,V,0,0,W,0
2,dehergne-b.cli,V,0,0,W,0
3,dehergne-c.cli,V,0,0,I,0
4,dehergne-d.cli,V,0,0,I,0
5,dehergne-e.cli,V,0,0,I,0
6,dehergne-f.cli,V,0,0,I,0
7,dehergne-g.cli,V,0,0,I,0
8,dehergne-h.cli,V,0,0,I,0
9,dehergne-i.cli,V,0,0,I,0


## Optionally clean translations 

Do this if there was an update to the Kleio server and you want to make sure that all translations are redone to incorporate the changes.

In [9]:
# NBVAL_IGNORE_OUTPUT

# tlnb.kleio_server.translation_clean("", recurse="yes")


### Translate and import as needed

In [10]:
tlnb.db.update_from_sources(path='', with_import_errors=True, match_path=True)
kleio_files = tlnb.get_kleio_files()
kleio_files[['name','status','errors','warnings','import_status','import_errors']]


,name,status,errors,warnings,import_status,import_errors
0,dehergne-0-abrev.cli,V,0,0,I,0
1,dehergne-a.cli,V,0,0,W,0
2,dehergne-b.cli,V,0,0,W,0
3,dehergne-c.cli,V,0,0,I,0
4,dehergne-d.cli,V,0,0,I,0
5,dehergne-e.cli,V,0,0,I,0
6,dehergne-f.cli,V,0,0,I,0
7,dehergne-g.cli,V,0,0,I,0
8,dehergne-h.cli,V,0,0,I,0
9,dehergne-i.cli,V,0,0,I,0


## Check the status of the files

Check the import status of the translated files

* I: Imported
* E: Imported with error
* W: Imported with warnings no errors
* N: Not imported
* U: Translation updated need to reimport

### Check a translation report

If needed to check for errors.


In [11]:
# get the translation status
imported_files_df = tlnb.get_import_status(match_path=True)
errors = imported_files_df[imported_files_df["errors"] > 0 ]
errors[["path","name", "errors", "warnings", "import_status", "import_errors", "import_warnings", "path"]].sort_values("name")

,path,name,errors,warnings,import_status,import_errors,import_warnings,path


In [12]:
imported_files_df[['name','import_status']]

,name,import_status
0,dehergne-0-abrev.cli,I
1,dehergne-a.cli,W
2,dehergne-b.cli,W
3,dehergne-c.cli,I
4,dehergne-d.cli,I
5,dehergne-e.cli,I
6,dehergne-f.cli,I
7,dehergne-g.cli,I
8,dehergne-h.cli,I
9,dehergne-i.cli,I


In [13]:
file_number=list(errors.index.unique())
if len(file_number) > 0:
    rpt = tlnb.get_translation_report(imported_files_df, file_number)
    print(rpt)
else:
    print("No errors found")



No errors found


### Get the import report for a file

In case there are errors in the import phase check the import report.

In [14]:
ierrors = imported_files_df[imported_files_df.import_errors>0][["import_status", "import_errors","import_warnings","name","imported","errors","warnings","path"]].sort_values("name")
ierrors

,import_status,import_errors,import_warnings,name,imported,errors,warnings,path


In [15]:
for ierror in ierrors.index:
    print(f"File: {ierrors.loc[ierror,'name']} - {ierrors.loc[ierror,'path']}")
    rpt = tlnb.get_import_rpt(imported_files_df, rows=ierror, match_path=True)
    print(rpt)

In [16]:
tlnb.get_import_rpt('dehergne-v.cli')  # just the file name of the kleio file

'No errors\nNo warnings'

In [17]:
from datetime import datetime

# Print current timestamp
print(datetime.now())

# Or with a specific format
print(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

# Or using other formats
print(f"Current time: {datetime.now()}")


2026-10-30 16:53:05.097326
2026-10-30 16:53:05
Current time: 2026-10-30 16:53:05.097623
